# Problem 3 — Impact of Noise on the Dynamic Long-Range CNOT

**2026 Quantum Hackathon — Dynamic Circuits**

Reference: Bäumer *et al.*, PRX Quantum **5**, 030339 (2024).

---

## Objective

Analyze how **depolarizing noise**, **readout error**, and **thermal relaxation (T₁/T₂)** degrade fidelity of the dynamic long-range CNOT as the number of intermediate ancilla qubits $n$ increases.

The error budget decomposes as:

$$\lambda_{\text{tot}} = t_{\text{idle}}\,\lambda_{\text{idle}} + N_{\text{CNOT}}\,\lambda_{\text{CNOT}} + N_{\text{meas}}\,\lambda_{\text{meas}}, \qquad F_{\text{proc}} \geq e^{-\lambda_{\text{tot}}}$$

In [ ]:
from __future__ import annotations

import math, gc, warnings
from collections import deque
from dataclasses import dataclass
from statistics import median

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister, transpile
from qiskit.circuit import IfElseOp, Reset
from qiskit.circuit.classical import expr
from qiskit.transpiler import CouplingMap, Layout, generate_preset_pass_manager
from qiskit_aer import AerSimulator
from qiskit_aer.noise import (
    NoiseModel, ReadoutError,
    depolarizing_error, thermal_relaxation_error,
)

warnings.filterwarnings("ignore", category=DeprecationWarning)
plt.style.use("seaborn-v0_8-whitegrid")

SHOTS = 4096
SEED = 2026
np.random.seed(SEED)
print("Imports complete.")

## Step 0 — Backend Selection

아래 셀에서 사용할 **Fake Backend**를 선택하세요. `BACKEND_NAME`을 바꾸면 다른 백엔드로 전환됩니다.

사용 가능한 백엔드 예시:
- `"FakeBrisbane"` — 127-qubit Eagle r3, heavy-hex
- `"FakeSherbrooke"` — 127-qubit Eagle r3, heavy-hex
- `"FakeKyoto"` — 127-qubit Eagle r3, heavy-hex
- `"FakeOsaka"` — 127-qubit Eagle r3, heavy-hex
- `"FakeTorino"` — 133-qubit Heron r1
- `"FakeQuebec"` — 127-qubit Eagle r1
- `"FakeMontreal"` — 27-qubit Falcon r4

> 원하는 백엔드 이름을 아래 `BACKEND_NAME`에 넣으면 자동으로 로드됩니다.

In [ ]:
# ============================================================
# >>>  여기서 백엔드를 선택하세요  <<<
# ============================================================
BACKEND_NAME = "FakeBrisbane"
# BACKEND_NAME = "FakeSherbrooke"
# BACKEND_NAME = "FakeKyoto"
# BACKEND_NAME = "FakeOsaka"
# BACKEND_NAME = "FakeTorino"
# BACKEND_NAME = "FakeQuebec"
# BACKEND_NAME = "FakeMontreal"

# ============================================================

from qiskit_ibm_runtime import fake_provider as _fp

_available = {name: getattr(_fp, name)
              for name in dir(_fp)
              if name.startswith("Fake") and name[4:5].isupper()
              and callable(getattr(_fp, name, None))}

if BACKEND_NAME not in _available:
    print(f"'{BACKEND_NAME}' not found. Available backends:")
    for k in sorted(_available):
        print(f"  {k}")
    raise ValueError(f"Unknown backend: {BACKEND_NAME}")

backend_ref = _available[BACKEND_NAME]()

# Ensure dynamic-circuit instruction support
for instr_cls, instr_name in [(IfElseOp, "if_else"), (Reset, "reset")]:
    if instr_name not in backend_ref.target.operation_names:
        backend_ref.target.add_instruction(instr_cls, name=instr_name)

print(f"Selected backend: {backend_ref.name}")
print(f"  Qubits: {backend_ref.num_qubits}")
print(f"  Native gates: {sorted(set(backend_ref.target.operation_names) - {'if_else','reset','measure','delay','barrier'})}")

In [ ]:
# ----- Distances -----
SIM_DISTANCES    = [0, 1, 2, 4, 6, 8, 10, 14, 18]
BUDGET_DISTANCES = list(range(0, 51))
BASES            = ["XX", "YY", "ZZ"]
IMPLEMENTATIONS  = ["dynamic", "unitary_ic"]

# ----- Feed-forward latency (ns) — paper reports ~0.7 us -----
FEEDFORWARD_TIME_NS = 700.0

# ----- Real backend toggle (Part 6) -----
USE_REAL_BACKEND = False

# ----- Reference budget parameters (paper Fig. 7) -----
DEFAULT_BUDGET_PARAMS = {
    "mu": 3.65,
    "lambda_idle": 0.03,
    "lambda_cnot": 0.02,
    "lambda_meas": 0.03,
}

print(f"Simulation distances: {SIM_DISTANCES}")
print(f"Budget distances: 0 .. {max(BUDGET_DISTANCES)}")

## Step 1 — Lambda Conversion Functions (Appendix G & C)

Backend calibration 데이터를 Pauli-Lindblad rate ($\lambda$)로 변환하는 핵심 함수들입니다.

**Idle decoherence** (Appendix G, Eq. G7):
$$\lambda_{\text{idle}} = t_{\text{CNOT}} \left(\frac{1}{2T_1} + \frac{1}{2T_2}\right)$$

**CNOT gate error** (Appendix C, Eq. C13):
$$F_{\text{proc}} = \frac{(d+1)F_{\text{gate}} - 1}{d}, \quad \lambda_{\text{CNOT}} = -\log F_{\text{proc}} \quad (d=4)$$

**Readout / measurement error** (Eq. G1):
$$\lambda_{\text{meas}} = -\log(1 - p)$$

In [ ]:
def _error_probability_to_lambda(probability: float) -> float:
    """Convert calibration error probability to Pauli-Lindblad rate.
    
    For readout error probability p, the exact Pauli-Lindblad rate is
    lambda = -log(1 - p)  (Eq. G1).
    """
    p = min(max(float(probability), 0.0), 1.0 - 1e-12)
    return float(-np.log(1.0 - p))


def _idle_lambda_per_cnot_time(t1_ns: float, t2_ns: float, cnot_time_ns: float) -> float:
    """Idle decoherence rate over one CNOT-gate duration (Appendix G, Eq. G7).
    
    Combined amplitude-damping (t/4T1 for X, t/4T1 for Y) and
    phase-damping (t/2T2 for Z) gives total idle rate per CNOT-time:
        lambda_idle = t_CNOT * (1/(2*T1) + 1/(2*T2))
    """
    if min(t1_ns, t2_ns, cnot_time_ns) <= 0:
        return DEFAULT_BUDGET_PARAMS["lambda_idle"]
    return float(cnot_time_ns * (1.0 / (2.0 * t1_ns) + 1.0 / (2.0 * t2_ns)))


def _cnot_lambda_per_operation(cnot_gate_error: float, dimension: int = 4) -> float:
    """Budget-effective CNOT rate from average gate error (Appendix C, Eq. C13).
    
    Qiskit gate_error is average gate error: F_gate = 1 - gate_error.
    Process fidelity: F_proc = ((d+1)*F_gate - 1) / d,  d=4 for 2-qubit.
    Budget rate: lambda_CNOT = -log(F_proc).
    """
    gate_error = min(max(float(cnot_gate_error), 0.0), 1.0 - 1e-12)
    gate_fidelity = 1.0 - gate_error
    process_fidelity = ((dimension + 1.0) * gate_fidelity - 1.0) / dimension
    process_fidelity = min(max(process_fidelity, 1e-12), 1.0)
    return float(-np.log(process_fidelity))


print("Lambda conversion functions defined.")

## Step 2 — Hardware Calibration Extraction

선택한 Fake Backend에서 T₁, T₂, gate error, readout error, gate duration 등을 추출하고, $\lambda$ rate로 변환합니다.

In [ ]:
TWO_Q_NAMES = {"cx", "ecr", "cz"}

def median_2q_duration(backend):
    durs = []
    for name in TWO_Q_NAMES:
        if name not in backend.target.operation_names: continue
        for _, p in backend.target[name].items():
            if p and p.duration: durs.append(p.duration)
    return float(median(durs)) if durs else 68e-9

def median_2q_error(backend):
    errs = []
    for name in TWO_Q_NAMES:
        if name not in backend.target.operation_names: continue
        for _, p in backend.target[name].items():
            if p and p.error is not None: errs.append(p.error)
    return float(median(errs)) if errs else 0.01

def median_readout_err(backend, qubits=None):
    if "measure" not in backend.target.operation_names: return 0.02
    qubits = qubits or list(range(min(30, len(backend.target.qubit_properties))))
    errs = []
    for q in qubits:
        p = backend.target["measure"].get((q,))
        if p and p.error is not None: errs.append(p.error)
    return float(median(errs)) if errs else 0.02

def median_t1_t2_us(backend, qubits=None):
    props = backend.target.qubit_properties
    qubits = qubits or list(range(len(props)))
    t1s = [props[q].t1 * 1e6 for q in qubits if props[q] and props[q].t1]
    t2s = [props[q].t2 * 1e6 for q in qubits if props[q] and props[q].t2]
    return float(median(t1s)) if t1s else 100.0, float(median(t2s)) if t2s else 50.0

def median_meas_time_ns(backend):
    if "measure" not in backend.target.operation_names: return 800.0
    durs = []
    for _, p in backend.target["measure"].items():
        if p and p.duration: durs.append(p.duration * 1e9)
    return float(median(durs)) if durs else 800.0


# ---- Extract from selected backend ----
t_2q_s   = median_2q_duration(backend_ref)
gate_err = median_2q_error(backend_ref)
ro_err   = median_readout_err(backend_ref)
t1_us, t2_us = median_t1_t2_us(backend_ref)
t_meas_ns = median_meas_time_ns(backend_ref)

# Convert to ns
t_2q_ns = t_2q_s * 1e9
t1_ns   = t1_us * 1e3
t2_ns   = min(t2_us * 1e3, 2.0 * t1_ns)

# Compute calibrated budget parameters
mu_cal          = (t_meas_ns + FEEDFORWARD_TIME_NS) / t_2q_ns
lambda_idle_cal = _idle_lambda_per_cnot_time(t1_ns, t2_ns, t_2q_ns)
lambda_cnot_cal = _cnot_lambda_per_operation(gate_err, dimension=4)
lambda_meas_cal = _error_probability_to_lambda(ro_err)

CALIBRATED_BUDGET = {
    "mu": mu_cal,
    "lambda_idle": lambda_idle_cal,
    "lambda_cnot": lambda_cnot_cal,
    "lambda_meas": lambda_meas_cal,
}

# Display
cal_df = pd.DataFrame([{
    "backend": backend_ref.name,
    "qubits": backend_ref.num_qubits,
    "T1 (us)": round(t1_us, 1),
    "T2 (us)": round(t2_us, 1),
    "t_2q (ns)": round(t_2q_ns, 1),
    "t_meas (ns)": round(t_meas_ns, 1),
    "gate_err": round(gate_err, 5),
    "readout_err": round(ro_err, 5),
}])

lam_df = pd.DataFrame([{
    "mu": round(mu_cal, 3),
    "lambda_idle": round(lambda_idle_cal, 6),
    "lambda_CNOT": round(lambda_cnot_cal, 6),
    "lambda_meas": round(lambda_meas_cal, 6),
    "3*lambda_CNOT": round(3*lambda_cnot_cal, 6),
    "crossover met?": lambda_meas_cal < 3*lambda_cnot_cal,
}])

print(f"=== {backend_ref.name} Hardware Calibration ===")
display(cal_df)
print(f"\n=== Derived Lambda Rates ===")
print(f"  lambda_idle  = t_CNOT*(1/2T1 + 1/2T2)     = {lambda_idle_cal:.6f}")
print(f"  lambda_CNOT  = -log(F_proc)                = {lambda_cnot_cal:.6f}  [F_gate={1-gate_err:.5f}]")
print(f"  lambda_meas  = -log(1-p_ro)                = {lambda_meas_cal:.6f}  [p_ro={ro_err:.5f}]")
print(f"  mu           = (t_meas+t_ff)/t_CNOT        = {mu_cal:.3f}")
display(lam_df)

## Step 3 — Circuit Builders

**Dynamic LRCX**: Gate teleportation, O(1) quantum depth.
**Unitary Ic**: Full round-trip SWAP, N_CNOT = 4n+1, t_idle = 0.

In [ ]:
# ==================== Dynamic LRCX ====================

def check_even(n: int) -> int:
    return 1 if n % 2 == 0 else 2

def initialize_circuit(distance: int) -> QuantumCircuit:
    n = distance
    qr = QuantumRegister(n + 2, name="q")
    cr = ClassicalRegister(2, name="cr")
    k = int(n / 2)
    allcr = [cr]
    if distance > 1:
        allcr.append(ClassicalRegister(k, name="c1"))
    if distance > 0:
        allcr.append(ClassicalRegister(n - k, name="c2"))
    qc = QuantumCircuit(qr, *allcr, name="LRCX")
    qc.h(0)
    return qc

def prepare_bell_pairs(qc):
    n = qc.num_qubits - 2; k = int(n / 2); x0 = check_even(n)
    if n % 2 != 0: qc.cx(0, 1)
    for i in range(k):
        qc.h(x0 + 2*i); qc.cx(x0 + 2*i, x0 + 2*i + 1)
    return qc

def measure_bell_basis(qc):
    n = qc.num_qubits - 2; k = int(n / 2)
    if n > 1: _, c1, c2 = qc.cregs
    elif n > 0: _, c2 = qc.cregs
    x0 = 1 if n % 2 == 0 else 2
    for i in range(k + 1): qc.cx(x0 - 1 + 2*i, x0 + 2*i)
    for i in range(1, k + x0): qc.h(2*i + 1 - x0)
    qc.barrier()
    for i in range(k): qc.measure(2*i + x0, c1[i])
    for i in range(1, k + x0): qc.measure(2*i + 1 - x0, c2[i - 1])
    for q in range(1, n + 1): qc.reset(q)
    return qc

def apply_ffwd_corrections(qc):
    control, target = 0, qc.num_qubits - 1
    n = qc.num_qubits - 2; k = int(n / 2); x0 = check_even(n)
    if n > 1: _, c1, c2 = qc.cregs
    elif n > 0: _, c2 = qc.cregs
    parity_ZZ = parity_XX = None
    for i in range(k):
        parity_ZZ = expr.lift(c1[i]) if i == 0 else expr.bit_xor(c1[i], parity_ZZ)
    for i in range(1, k + x0):
        parity_XX = expr.lift(c2[i-1]) if i == 1 else expr.bit_xor(c2[i-1], parity_XX)
    if n > 0 and parity_XX is not None:
        with qc.if_test(parity_XX): qc.z(control)
    if n > 1 and parity_ZZ is not None:
        with qc.if_test(parity_ZZ): qc.x(target)
    return qc

def build_dynamic_lrcx(distance):
    qc = initialize_circuit(distance)
    qc = prepare_bell_pairs(qc)
    qc = measure_bell_basis(qc)
    qc = apply_ffwd_corrections(qc)
    return qc


# ==================== Unitary Ic ====================

def build_unitary_cnot_ic(distance):
    n = distance
    qr = QuantumRegister(n + 2, name="q")
    cr = ClassicalRegister(2, name="cr")
    qc = QuantumCircuit(qr, cr, name="CNOT_Ic")
    qc.h(0); qc.barrier()
    for i in range(n + 1): qc.cx(i, i + 1)
    for i in range(n - 1, -1, -1): qc.cx(i, i + 1)
    for i in range(n, 0, -1): qc.cx(i, i + 1)
    for i in range(1, n + 1): qc.cx(i, i + 1)
    return qc


# ==================== Bell-test wrappers ====================

def _measure_in_basis(qc, basis):
    qc = qc.copy(); cr = qc.cregs[0]
    ctrl, tgt = 0, qc.num_qubits - 1; qc.barrier()
    if basis == "XX": qc.h(ctrl); qc.h(tgt)
    elif basis == "YY": qc.sdg(ctrl); qc.sdg(tgt); qc.h(ctrl); qc.h(tgt)
    qc.measure(ctrl, cr[0]); qc.measure(tgt, cr[1])
    return qc

def build_bell_test_circuits(distance, impl):
    if impl == "dynamic": core = build_dynamic_lrcx(distance)
    elif impl == "unitary_ic": core = build_unitary_cnot_ic(distance)
    else: raise ValueError(impl)
    return [_measure_in_basis(core, b) for b in BASES]

# Sanity check
for impl in IMPLEMENTATIONS:
    circs = build_bell_test_circuits(2, impl)
    sim = AerSimulator(seed_simulator=SEED)
    res = sim.run(circs, shots=1024).result()
    counts = [res.get_counts(i) for i in range(3)]
    def _zz(c):
        t = sum(c.values())
        return sum(((-1)**(int(b[-1])+int(b[-2])))*v for b,v in c.items()) / t
    F = 0.25 * (1 + _zz(counts[0]) - _zz(counts[1]) + _zz(counts[2]))
    print(f"  {impl:12s} ideal Bell fidelity = {F:.4f}")
print("Circuit builders ready.")

### Example Circuits (distance n = 4)

In [ ]:
# Dynamic LRCX (gate teleportation, O(1) depth)
qc_dynamic = build_dynamic_lrcx(4)
print(f"Dynamic LRCX (n=4): {dict(qc_dynamic.count_ops())}")
print(f"  qubits={qc_dynamic.num_qubits}, depth={qc_dynamic.depth()}")
qc_dynamic.draw("mpl", fold=-1, scale=0.55,
                style={"backgroundcolor": "#EEEEEE"})

In [ ]:
# Unitary Ic (zero-idle full SWAP, O(n) depth)
qc_unitary = build_unitary_cnot_ic(4)
print(f"Unitary Ic (n=4): {dict(qc_unitary.count_ops())}")
print(f"  qubits={qc_unitary.num_qubits}, depth={qc_unitary.depth()}")
qc_unitary.draw("mpl", fold=-1, scale=0.55,
                style={"backgroundcolor": "#EEEEEE"})

## Part 1 — Noise Models (Generic Aer Sweep용)

| Model | Description |
|---|---|
| `depolarizing` | 1Q + 2Q depolarizing |
| `readout` | Symmetric bit-flip |
| `thermal` | T₁/T₂ relaxation |
| `combined` | All composed |

여기서는 **백엔드 캘리브레이션 값**을 사용합니다.

In [ ]:
# Noise parameters FROM the selected backend calibration
NOISE_PARAMS = {
    "p_depol_1q": 1.0e-4,       # 1Q gate error (small)
    "p_depol_2q": gate_err,     # from backend
    "p_readout":  ro_err,       # from backend
    "t1":   t1_ns,              # ns, from backend
    "t2":   t2_ns,              # ns, from backend
    "time_1q":   50,            # ns
    "time_2q":  t_2q_ns,        # ns, from backend
    "time_meas": t_meas_ns,     # ns, from backend
    "time_reset": t_meas_ns,    # ns
}

ONE_QUBIT_GATES = ["x", "z", "h", "sx", "id"]
TWO_QUBIT_GATES = ["cx"]

def two_qubit_thermal(t1, t2, gt):
    e = thermal_relaxation_error(t1, t2, gt); return e.tensor(e)

def make_noise_models(p):
    depol = NoiseModel()
    depol.add_all_qubit_quantum_error(depolarizing_error(p["p_depol_1q"], 1), ONE_QUBIT_GATES)
    depol.add_all_qubit_quantum_error(depolarizing_error(p["p_depol_2q"], 2), TWO_QUBIT_GATES)
    ro = NoiseModel()
    ro_err_mat = ReadoutError([[1-p["p_readout"],p["p_readout"]],[p["p_readout"],1-p["p_readout"]]])
    ro.add_all_qubit_readout_error(ro_err_mat)
    therm = NoiseModel()
    therm.add_all_qubit_quantum_error(thermal_relaxation_error(p["t1"],p["t2"],p["time_1q"]), ONE_QUBIT_GATES)
    therm.add_all_qubit_quantum_error(two_qubit_thermal(p["t1"],p["t2"],p["time_2q"]), TWO_QUBIT_GATES)
    therm.add_all_qubit_quantum_error(thermal_relaxation_error(p["t1"],p["t2"],p["time_meas"]), ["measure"])
    therm.add_all_qubit_quantum_error(thermal_relaxation_error(p["t1"],p["t2"],p["time_reset"]), ["reset"])
    combo = NoiseModel()
    combo.add_all_qubit_quantum_error(
        depolarizing_error(p["p_depol_1q"],1).compose(thermal_relaxation_error(p["t1"],p["t2"],p["time_1q"])),
        ONE_QUBIT_GATES)
    combo.add_all_qubit_quantum_error(
        depolarizing_error(p["p_depol_2q"],2).compose(two_qubit_thermal(p["t1"],p["t2"],p["time_2q"])),
        TWO_QUBIT_GATES)
    combo.add_all_qubit_quantum_error(thermal_relaxation_error(p["t1"],p["t2"],p["time_meas"]), ["measure"])
    combo.add_all_qubit_quantum_error(thermal_relaxation_error(p["t1"],p["t2"],p["time_reset"]), ["reset"])
    combo.add_all_qubit_readout_error(ro_err_mat)
    return {"ideal": None, "depolarizing": depol, "readout": ro, "thermal": therm, "combined": combo}

noise_models = make_noise_models(NOISE_PARAMS)
print(f"Noise models built from {backend_ref.name} calibration: {list(noise_models.keys())}")
print(f"  p_depol_2q={NOISE_PARAMS['p_depol_2q']:.4f}, p_readout={NOISE_PARAMS['p_readout']:.4f}")
print(f"  T1={NOISE_PARAMS['t1']:.0f} ns, T2={NOISE_PARAMS['t2']:.0f} ns, t_2q={NOISE_PARAMS['time_2q']:.0f} ns")

## Part 2 — Fidelity Estimation

$$F = \frac{1}{4}(1 + \langle XX\rangle - \langle YY\rangle + \langle ZZ\rangle)$$

In [ ]:
def zz_expectation(counts):
    total = sum(counts.values())
    if total == 0: return 0.0
    return sum(((-1)**(int(b.replace(' ','')[-1])+int(b.replace(' ','')[-2])))*v
               for b,v in counts.items()) / total

def bell_fidelity(cx, cy, cz):
    return 0.25 * (1.0 + zz_expectation(cx) - zz_expectation(cy) + zz_expectation(cz))

def run_fidelity(n_ancilla, impl, noise_model, shots=SHOTS):
    circs = build_bell_test_circuits(n_ancilla, impl)
    sim = AerSimulator(noise_model=noise_model, seed_simulator=SEED)
    res = sim.run(circs, shots=shots).result()
    return bell_fidelity(res.get_counts(0), res.get_counts(1), res.get_counts(2))

print("Fidelity helpers ready.")

## Part 3 — Direct Noisy Simulation (Aer Sweep)

각 noise model 별로 fidelity 측정 → **dominant limiting factor** 파악.

In [ ]:
def run_noise_sweep(distances, implementations, noise_models_dict):
    rows = []
    for impl in implementations:
        for noise_name, model in noise_models_dict.items():
            for n in distances:
                F = run_fidelity(n, impl, model)
                rows.append({"implementation": impl, "noise_model": noise_name,
                             "n_ancilla": n, "fidelity": F, "infidelity": 1.0 - F})
                print(f"  {impl:12s} {noise_name:13s} n={n:2d}  F={F:.4f}")
    return pd.DataFrame(rows)

print(f"Running noise sweep (backend params from {backend_ref.name})...")
results_df = run_noise_sweep(SIM_DISTANCES, IMPLEMENTATIONS, noise_models)
print(f"Done. {len(results_df)} data points.")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)
style_map = {"ideal":("o-","tab:gray"), "depolarizing":("s-","tab:blue"),
             "readout":("^-","tab:orange"), "thermal":("D-","tab:green"),
             "combined":("*-","tab:red")}
for ax, impl in zip(axes, IMPLEMENTATIONS):
    sub = results_df[results_df["implementation"] == impl]
    for nn, grp in sub.groupby("noise_model", sort=False):
        s,c = style_map.get(nn, ("o-","k"))
        ax.plot(grp["n_ancilla"], grp["fidelity"], s, color=c, label=nn, markersize=6)
    ax.set_xlabel("n (intermediate ancilla qubits)"); ax.set_title(impl.replace("_"," ").title())
    ax.set_ylim(0, 1.05); ax.axhline(0.5, color="gray", ls=":", alpha=0.5)
axes[0].set_ylabel("Bell-state fidelity"); axes[1].legend(loc="best", fontsize=9)
fig.suptitle(f"Aer Noise Sweep ({backend_ref.name} calibration)", fontsize=14, y=1.02)
plt.tight_layout(); plt.show()

In [ ]:
max_n = max(SIM_DISTANCES)
single = results_df[results_df["noise_model"].isin(["depolarizing","readout","thermal"])]
at_max = single[single["n_ancilla"]==max_n].sort_values(["implementation","infidelity"], ascending=[True,False])
print(f"=== Dominant noise at n={max_n} ({backend_ref.name}) ===\n")
for impl in IMPLEMENTATIONS:
    row = at_max[at_max["implementation"]==impl].iloc[0]
    print(f"  {impl:12s}: {row.noise_model} (infidelity={row.infidelity:.4f})")

## Part 4 — Analytical Error Budget (Table I)

> **이제 선택한 하드웨어 스펙이 반영된 $\lambda$ 값을 사용합니다.**

| Implementation | $t_{\text{idle}}$ | $N_{\text{CNOT}}$ | $N_{\text{meas}}$ | 2Q depth |
|---|---|---|---|---|
| **Unitary Ic** | 0 | $4n+1$ | 0 | $2n+1$ |
| **Dynamic** | $2\mu+2$ | $n+1$ | $n$ | $2+\mu$ |

Crossover condition: $\lambda_{\text{meas}} < 3\lambda_{\text{CNOT}}$

In [ ]:
def table_i_resources(n, impl, mu):
    if impl == "unitary_ic":
        return {"t_idle": 0.0, "N_CNOT": 4*n+1, "N_meas": 0, "depth_2q": 2*n+1}
    elif impl == "dynamic":
        return {"t_idle": 2*mu+2, "N_CNOT": n+1, "N_meas": n, "depth_2q": 2+mu}
    raise ValueError(impl)

def compute_lambda_tot(resources, params):
    return (resources["t_idle"] * params["lambda_idle"]
          + resources["N_CNOT"] * params["lambda_cnot"]
          + resources["N_meas"] * params["lambda_meas"])

# ==================================================================
# Build budget with BOTH reference and calibrated parameters
# ==================================================================
budget_rows = []
for label, params in [("reference (paper)", DEFAULT_BUDGET_PARAMS),
                      (f"calibrated ({backend_ref.name})", CALIBRATED_BUDGET)]:
    for impl in IMPLEMENTATIONS:
        for n in BUDGET_DISTANCES:
            res = table_i_resources(n, impl, params["mu"])
            lam = compute_lambda_tot(res, params)
            budget_rows.append({
                "param_source": label,
                "implementation": impl,
                "n_ancilla": n,
                **res,
                "lambda_idle_term":  res["t_idle"] * params["lambda_idle"],
                "lambda_cnot_term":  res["N_CNOT"] * params["lambda_cnot"],
                "lambda_meas_term":  res["N_meas"] * params["lambda_meas"],
                "lambda_total": lam,
                "F_lower_bound": np.exp(-lam),
            })

budget_df = pd.DataFrame(budget_rows)

# Crossover for calibrated params
cal_pivot = budget_df[budget_df["param_source"].str.contains("calibrated")].pivot(
    index="n_ancilla", columns="implementation", values="F_lower_bound").reset_index()
cal_pivot["dynamic_wins"] = cal_pivot["dynamic"] > cal_pivot["unitary_ic"]
cross_cal = cal_pivot[cal_pivot["dynamic_wins"]]
crossover_cal = None if cross_cal.empty else int(cross_cal.iloc[0]["n_ancilla"])

# Crossover for reference params
ref_pivot = budget_df[budget_df["param_source"].str.contains("reference")].pivot(
    index="n_ancilla", columns="implementation", values="F_lower_bound").reset_index()
ref_pivot["dynamic_wins"] = ref_pivot["dynamic"] > ref_pivot["unitary_ic"]
cross_ref = ref_pivot[ref_pivot["dynamic_wins"]]
crossover_ref = None if cross_ref.empty else int(cross_ref.iloc[0]["n_ancilla"])

print(f"=== Error Budget Summary ===")
print(f"  Reference params:   crossover at n = {crossover_ref}")
print(f"  Calibrated params:  crossover at n = {crossover_cal}  ({backend_ref.name})")
print(f"\n  Calibrated: lambda_meas={lambda_meas_cal:.5f}, 3*lambda_CNOT={3*lambda_cnot_cal:.5f}")
print(f"  -> crossover condition met: {lambda_meas_cal < 3*lambda_cnot_cal}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5.5))

# --- Left: Fidelity lower bound ---
ax = axes[0]
colors = {"dynamic": "tab:red", "unitary_ic": "tab:blue"}
for src_label, ls in [("reference (paper)", ":"), (f"calibrated ({backend_ref.name})", "-")]:
    sub = budget_df[budget_df["param_source"] == src_label]
    for impl in IMPLEMENTATIONS:
        g = sub[sub["implementation"] == impl]
        label = f"{impl.replace('_',' ')} [{src_label.split('(')[0].strip()}]"
        ax.plot(g["n_ancilla"], g["F_lower_bound"], ls, color=colors[impl], lw=2, label=label)

ax.set_xlabel("n"); ax.set_ylabel(r"$e^{-\lambda_{\mathrm{tot}}}$")
ax.set_ylim(0, 1.05); ax.set_title(f"Fidelity Lower Bound (Table I)")
ax.legend(fontsize=8)
if crossover_cal is not None:
    ax.axvline(crossover_cal, color="k", ls="--", alpha=0.5)
    ax.text(crossover_cal+0.5, 0.92, f"cal. crossover n={crossover_cal}", fontsize=9)
if crossover_ref is not None:
    ax.axvline(crossover_ref, color="gray", ls=":", alpha=0.5)

# --- Right: Component breakdown (calibrated only) ---
ax = axes[1]
cal_budget = budget_df[budget_df["param_source"].str.contains("calibrated")]
dyn = cal_budget[cal_budget["implementation"]=="dynamic"]
uni = cal_budget[cal_budget["implementation"]=="unitary_ic"]

ax.plot(uni["n_ancilla"], uni["lambda_total"], "--", color="tab:blue", lw=2, label="unitary Ic (CNOT only)")
ax.plot(dyn["n_ancilla"], dyn["lambda_total"], "-", color="tab:red", lw=2, label="dynamic: total")
ax.stackplot(dyn["n_ancilla"],
             dyn["lambda_idle_term"], dyn["lambda_cnot_term"], dyn["lambda_meas_term"],
             labels=["idle", "CNOT", "measurement"], alpha=0.25,
             colors=["gold","salmon","lightgreen"])
ax.set_xlabel("n"); ax.set_ylabel(r"$\lambda_{\mathrm{tot}}$")
ax.set_title(f"Component Breakdown ({backend_ref.name})")
ax.legend(loc="upper left", fontsize=8)

fig.suptitle(f"Analytical Error Budget — {backend_ref.name}", fontsize=14, y=1.02)
plt.tight_layout(); plt.show()

## Part 5 — Fake-Backend Transpiled Simulation

Fake backend의 coupling map 위에 transpile하고, `AerSimulator.from_backend`으로 시뮬레이션합니다.

In [ ]:
def find_connected_chain(coupling_map, length):
    num = coupling_map.size()
    nbrs = {i: set() for i in range(num)}
    for u, v in coupling_map.get_edges():
        nbrs[u].add(v); nbrs[v].add(u)
    found = []
    def dfs(path, visited):
        nonlocal found
        if len(path) >= length: found = path[:length]; return True
        for nb in sorted(nbrs[path[-1]]):
            if nb in visited: continue
            visited.add(nb); path.append(nb)
            if dfs(path, visited): return True
            path.pop(); visited.remove(nb)
        return False
    for s in range(num):
        if dfs([s], {s}): return found
    raise ValueError(f"No chain of length {length}")

def get_bus_chain(backend, n_logical):
    return find_connected_chain(backend.coupling_map, n_logical)

def transpile_on_chain(qc, backend, chain, opt_level=1):
    layout = Layout({qc.qubits[i]: chain[i] for i in range(qc.num_qubits)})
    pm = generate_preset_pass_manager(optimization_level=opt_level, backend=backend, initial_layout=layout)
    return pm.run(qc)

FAKE_SIM_DISTANCES = [0, 1, 2, 4, 6, 8, 10, 14]

def run_fake_backend_sweep(backend, distances, shots=2048):
    sim = AerSimulator.from_backend(backend)
    sim.set_options(max_parallel_experiments=1, seed_simulator=SEED)
    rows = []
    for impl in IMPLEMENTATIONS:
        for n in distances:
            try: chain = get_bus_chain(backend, n + 2)
            except ValueError: print(f"  skip {impl} n={n}"); continue
            circs = build_bell_test_circuits(n, impl)
            isa = [transpile_on_chain(c, backend, chain) for c in circs]
            cl = [sim.run(c, shots=shots).result().get_counts() for c in isa]
            F = bell_fidelity(cl[0], cl[1], cl[2])
            res = table_i_resources(n, impl, CALIBRATED_BUDGET["mu"])
            lam = compute_lambda_tot(res, CALIBRATED_BUDGET)
            rows.append({"implementation": impl, "n_ancilla": n,
                         "fidelity_sim": F, "F_budget": np.exp(-lam), "lambda_total": lam})
            print(f"  {impl:12s} n={n:2d}  F_sim={F:.4f}  F_budget={np.exp(-lam):.4f}")
    return pd.DataFrame(rows)

print(f"Running transpiled sweep on {backend_ref.name}...")
fake_results = run_fake_backend_sweep(backend_ref, FAKE_SIM_DISTANCES)
print(f"Done. {len(fake_results)} points.")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, impl in zip(axes, IMPLEMENTATIONS):
    sub = fake_results[fake_results["implementation"]==impl]
    ax.plot(sub["n_ancilla"], sub["fidelity_sim"], "o-", color="tab:red", markersize=7, label="Aer sim (transpiled)")
    ax.plot(sub["n_ancilla"], sub["F_budget"], "s--", color="tab:blue", markersize=7, label=r"Budget $e^{-\lambda}$ (cal.)")
    ax.set_xlabel("n"); ax.set_title(impl.replace("_"," ").title())
    ax.set_ylim(0, 1.05); ax.axhline(0.5, color="gray", ls=":", alpha=0.5); ax.legend(fontsize=9)
axes[0].set_ylabel("Fidelity")
fig.suptitle(f"{backend_ref.name}: Transpiled Simulation vs Calibrated Budget", fontsize=14, y=1.02)
plt.tight_layout(); plt.show()

### Per-Noise-Source Breakdown on Fake Backend

In [ ]:
def build_selective_noise(backend, thermal=True, depol=True, readout=True):
    nm = NoiseModel(); target = backend.target
    for gn in ["id","sx","x","cx","ecr","cz","reset"]:
        if gn not in target.operation_names: continue
        for qargs, props in target[gn].items():
            if not qargs or not props: continue
            chs = []
            if thermal and props.duration and props.duration > 0:
                try:
                    tes = []
                    for q in qargs:
                        qp = target.qubit_properties[q]
                        _t1 = qp.t1 if qp.t1 else 100e-6
                        _t2 = min(qp.t2 if qp.t2 else 50e-6, 2*_t1)
                        tes.append(thermal_relaxation_error(_t1, _t2, props.duration))
                    c = tes[0]
                    for e in tes[1:]: c = c.tensor(e)
                    chs.append(c)
                except: pass
            if depol and props.error and props.error > 0:
                try:
                    nq = len(qargs); mx = 4**nq/(4**nq-1)
                    chs.append(depolarizing_error(min(props.error, mx-1e-10), nq))
                except: pass
            if not chs: continue
            c = chs[0]
            for ch in chs[1:]: c = c.compose(ch)
            try: nm.add_quantum_error(c, [gn], list(qargs), warnings=False)
            except: pass
    if readout and "measure" in target.operation_names:
        for qargs, props in target["measure"].items():
            if not qargs or len(qargs)!=1 or not props or props.error is None: continue
            p = min(max(props.error,0),0.5)
            nm.add_readout_error(ReadoutError([[1-p,p],[p,1-p]]), [qargs[0]])
    return nm

configs = {"thermal only":{"thermal":True,"depol":False,"readout":False},
           "gate error only":{"thermal":False,"depol":True,"readout":False},
           "readout only":{"thermal":False,"depol":False,"readout":True},
           "all combined":{"thermal":True,"depol":True,"readout":True}}
BD = [0,2,4,6,8,10]
bd_rows = []
for lbl, cfg in configs.items():
    nm = build_selective_noise(backend_ref, **cfg)
    sim = AerSimulator(noise_model=nm, coupling_map=backend_ref.coupling_map,
                       basis_gates=["cx","id","rz","sx","x","reset","measure","if_else"],
                       seed_simulator=SEED)
    for n in BD:
        try: chain = get_bus_chain(backend_ref, n+2)
        except: continue
        isa = [transpile_on_chain(c, backend_ref, chain) for c in build_bell_test_circuits(n,"dynamic")]
        cl = [sim.run(c, shots=2048).result().get_counts() for c in isa]
        F = bell_fidelity(cl[0],cl[1],cl[2])
        bd_rows.append({"noise_source":lbl,"n_ancilla":n,"fidelity":F,"infidelity":1-F})
        print(f"  {lbl:20s} n={n:2d}  F={F:.4f}")
breakdown_df = pd.DataFrame(bd_rows)
print("Done.")

In [ ]:
fig, ax = plt.subplots(figsize=(9,5))
sty = {"thermal only":("D-","tab:green"), "gate error only":("s-","tab:blue"),
       "readout only":("^-","tab:orange"), "all combined":("*-","tab:red")}
for lbl, grp in breakdown_df.groupby("noise_source", sort=False):
    s,c = sty.get(lbl,("o-","k"))
    ax.plot(grp["n_ancilla"], grp["fidelity"], s, color=c, label=lbl, markersize=8)
ax.set_xlabel("n"); ax.set_ylabel("Dynamic LRCX fidelity")
ax.set_title(f"Noise Source Breakdown ({backend_ref.name})"); ax.set_ylim(0,1.05)
ax.axhline(0.5, color="gray", ls=":", alpha=0.4); ax.legend(fontsize=10)
plt.tight_layout(); plt.show()

mx = max(BD)
at_mx = breakdown_df[(breakdown_df["n_ancilla"]==mx)&(breakdown_df["noise_source"]!="all combined")]
at_mx = at_mx.sort_values("infidelity", ascending=False)
print(f"\nDominant noise at n={mx}: {at_mx.iloc[0]['noise_source']} (infidelity={at_mx.iloc[0]['infidelity']:.4f})")

## Part 6 — Real Hardware (Optional)

`USE_REAL_BACKEND = True`로 설정하면 IBM Quantum 백엔드에서 캘리브레이션을 가져옵니다.

In [ ]:
if USE_REAL_BACKEND:
    from qiskit_ibm_runtime import QiskitRuntimeService
    service = QiskitRuntimeService()
    real_backend = service.least_busy(operational=True, simulator=False, min_num_qubits=127)
    print(f"Real backend: {real_backend.name}")
    _t2q = median_2q_duration(real_backend); _ge = median_2q_error(real_backend)
    _ro = median_readout_err(real_backend); _t1r,_t2r = median_t1_t2_us(real_backend)
    _t2q_ns = _t2q*1e9; _t1r_ns = _t1r*1e3; _t2r_ns = min(_t2r*1e3, 2*_t1r_ns)
    REAL_BUDGET = {
        "mu": (median_meas_time_ns(real_backend)+FEEDFORWARD_TIME_NS)/_t2q_ns,
        "lambda_idle": _idle_lambda_per_cnot_time(_t1r_ns, _t2r_ns, _t2q_ns),
        "lambda_cnot": _cnot_lambda_per_operation(_ge, 4),
        "lambda_meas": _error_probability_to_lambda(_ro),
    }
    display(pd.DataFrame([{"backend":real_backend.name, **REAL_BUDGET}]))
else:
    print("USE_REAL_BACKEND=False. Using fake backend calibration as proxy.")
    REAL_BUDGET = CALIBRATED_BUDGET.copy()

### Three-Way Comparison

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))
for ax, impl in zip(axes, IMPLEMENTATIONS):
    ns = np.array(BUDGET_DISTANCES)
    # Reference budget
    F_ref = [np.exp(-compute_lambda_tot(table_i_resources(n,impl,DEFAULT_BUDGET_PARAMS["mu"]),DEFAULT_BUDGET_PARAMS)) for n in ns]
    ax.plot(ns, F_ref, "-", color="tab:gray", lw=2, alpha=0.6, label="Reference (paper Fig.7)")
    # Calibrated budget
    F_cal = [np.exp(-compute_lambda_tot(table_i_resources(n,impl,REAL_BUDGET["mu"]),REAL_BUDGET)) for n in ns]
    bl = "real HW" if USE_REAL_BACKEND else backend_ref.name
    ax.plot(ns, F_cal, "--", color="tab:green", lw=2, label=f"Calibrated ({bl})")
    # Fake-backend sim
    sub = fake_results[fake_results["implementation"]==impl]
    if not sub.empty:
        ax.plot(sub["n_ancilla"], sub["fidelity_sim"], "o", color="tab:red", ms=8, label=f"Aer sim ({backend_ref.name})")
    # Generic Aer sim
    gen = results_df[(results_df["implementation"]==impl)&(results_df["noise_model"]=="combined")]
    if not gen.empty:
        ax.plot(gen["n_ancilla"], gen["fidelity"], "^", color="tab:purple", ms=7, alpha=0.7, label="Aer sim (generic)")
    ax.set_xlabel("n"); ax.set_title(impl.replace("_"," ").title())
    ax.set_ylim(0, 1.05); ax.axhline(0.5, color="gray", ls=":", alpha=0.4)
    ax.legend(fontsize=8, loc="upper right")
axes[0].set_ylabel("Fidelity / Lower bound")
fig.suptitle("Three-Way Comparison: Budget vs Simulation", fontsize=14, y=1.02)
plt.tight_layout(); plt.show()

## Part 7 — Error Budget Consistency

In [ ]:
rows = []
for impl in IMPLEMENTATIONS:
    for _, row in fake_results[fake_results["implementation"]==impl].iterrows():
        n = int(row["n_ancilla"]); res = table_i_resources(n, impl, CALIBRATED_BUDGET["mu"])
        it = res["t_idle"]*CALIBRATED_BUDGET["lambda_idle"]
        ct = res["N_CNOT"]*CALIBRATED_BUDGET["lambda_cnot"]
        mt = res["N_meas"]*CALIBRATED_BUDGET["lambda_meas"]
        lt = it+ct+mt
        rows.append({"impl":impl, "n":n, "t_idle":res["t_idle"],
                     "N_CNOT":int(res["N_CNOT"]), "N_meas":int(res["N_meas"]),
                     "lam_idle":round(it,4), "lam_CNOT":round(ct,4),
                     "lam_meas":round(mt,4), "lam_tot":round(lt,4),
                     "F_budget":round(np.exp(-lt),4), "F_sim":round(row["fidelity_sim"],4)})
display(pd.DataFrame(rows))

## Discussion

### Dominant noise source

- **Gate error (depolarizing)**: Dynamic CNOT의 주요 오류원. n+1개의 2Q gate에서 누적.
- **Readout error**: n개의 mid-circuit measurement에서 기여. 두 번째로 중요.
- **Thermal relaxation**: Feed-forward idle time 동안 발생하나, O(1) depth로 상수 offset.

### Error budget 일관성

$$\lambda_{\text{tot}} = t_{\text{idle}}\lambda_{\text{idle}} + N_{\text{CNOT}}\lambda_{\text{CNOT}} + N_{\text{meas}}\lambda_{\text{meas}}$$

- **Unitary Ic**: $(4n+1)\lambda_{\text{CNOT}}$ — slope 4배 빠름
- **Dynamic**: $(n+1)\lambda_{\text{CNOT}} + n\lambda_{\text{meas}} + (2\mu+2)\lambda_{\text{idle}}$
- **Crossover**: $\lambda_{\text{meas}} < 3\lambda_{\text{CNOT}}$일 때 dynamic이 유리

### Key insight

Mid-circuit measurement error 감소가 dynamic circuit 경쟁력에 가장 큰 영향. Analytical budget $e^{-\lambda}$는 lower bound (Lemma 1)이며, 실제 simulation보다 보수적.

## Appendix: Parameter Summary

In [ ]:
summary = pd.DataFrame({
    "Parameter": ["lambda_idle","lambda_CNOT","lambda_meas","mu","3*lambda_CNOT","crossover?"],
    "Reference (paper)": [
        f"{DEFAULT_BUDGET_PARAMS['lambda_idle']:.4f}",
        f"{DEFAULT_BUDGET_PARAMS['lambda_cnot']:.4f}",
        f"{DEFAULT_BUDGET_PARAMS['lambda_meas']:.4f}",
        f"{DEFAULT_BUDGET_PARAMS['mu']:.2f}",
        f"{3*DEFAULT_BUDGET_PARAMS['lambda_cnot']:.4f}",
        str(DEFAULT_BUDGET_PARAMS['lambda_meas'] < 3*DEFAULT_BUDGET_PARAMS['lambda_cnot']),
    ],
    f"Calibrated ({backend_ref.name})": [
        f"{CALIBRATED_BUDGET['lambda_idle']:.6f}",
        f"{CALIBRATED_BUDGET['lambda_cnot']:.6f}",
        f"{CALIBRATED_BUDGET['lambda_meas']:.6f}",
        f"{CALIBRATED_BUDGET['mu']:.3f}",
        f"{3*CALIBRATED_BUDGET['lambda_cnot']:.6f}",
        str(CALIBRATED_BUDGET['lambda_meas'] < 3*CALIBRATED_BUDGET['lambda_cnot']),
    ],
})
display(summary)

print(f"\n=== Table I Resource Counts ===")
for impl in IMPLEMENTATIONS:
    for n in [0, 5, 10, 20, 40]:
        r = table_i_resources(n, impl, CALIBRATED_BUDGET["mu"])
        print(f"  {impl:12s} n={n:2d}:  t_idle={r['t_idle']:6.1f}  "
              f"N_CNOT={r['N_CNOT']:3.0f}  N_meas={r['N_meas']:2.0f}  depth_2q={r['depth_2q']:5.1f}")